# CMI Behavior Detection with Multimodal V40 (ws64 only)

This notebook uses a single 5-fold multimodal model (window size 64) for gesture recognition.

In [1]:
import os
from pathlib import Path
import polars as pl

import kaggle_evaluation.cmi_inference_server

from src.inference_pipeline import predict_one


2025-07-22 09:02:38.164453: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-22 09:02:39.573434: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753142560.060496  608809 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753142560.185875  608809 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753142561.290751  608809 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
def predict(sequence: pl.DataFrame, demographics: pl.DataFrame) -> str:
    return predict_one(sequence, demographics)


In [3]:
inference_server = kaggle_evaluation.cmi_inference_server.CMIInferenceServer(predict)

def is_kaggle():
    return "KAGGLE_URL_BASE" in os.environ or Path("/kaggle").exists()

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    if is_kaggle():
        inference_server.run_local_gateway(
            data_paths=(
                '/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv',
                '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv',
            )
        )
    else:
        inference_server.run_local_gateway(
            data_paths=(
                '../data/split/test.csv',
                '../data/split/test_demographics.csv',
            )
        )


INFO:src.inference_pipeline:Starting ensemble inference for one sample.
INFO:src.inference_pipeline:Processing with ws64 preprocessor...
INFO:utils.pipeline:Transforming dataframe of shape (79, 343)
/mnt/c/Users/ShunK/works/CMI_comp/submissions/v40/src/utils/feature_engineering.py:171: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[flag] = df[cols].isna().all(axis=1)
/mnt/c/Users/ShunK/works/CMI_comp/submissions/v40/src/utils/feature_engineering.py:171: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[flag] = df[cols].isna().a

interp:   0%|          | 0/1 [00:00<?, ?seq/s]

concat:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:utils.pipeline:Building windows (size=64, stride=32, min_len=20)
INFO:utils.pipeline:Windows shapes: X_sensor=(1, 64, 18), X_demo=(1, 7), y=(1,)
INFO:utils.pipeline:Window tensor shape (1, 64, 18)
INFO:utils.pipeline:Building tabular features (wavelet=True, tda=True, tof_event=False, temp_grad=False)
INFO:utils.pipeline:Tabular features shape (1, 409)
ERROR:src.inference_pipeline:Error in ws64 processing: X has 409 features, but StandardScaler is expecting 392 features as input.
ERROR:grpc._server:Exception calling application: X has 409 features, but StandardScaler is expecting 392 features as input.
Traceback (most recent call last):
  File "/mnt/c/Users/ShunK/works/CMI_comp/.venv/lib/python3.11/site-packages/grpc/_server.py", line 610, in _call_behavior
    response_or_iterator = behavior(argument, context)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/ShunK/works/CMI_comp/submissions/v40/kaggle_evaluation/core/relay.py", line 354, in Send
    resp

✅ TDA処理前: NaN値なし


GatewayRuntimeError: (<GatewayRuntimeErrorType.SERVER_RAISED_EXCEPTION: 3>, 'X has 409 features, but StandardScaler is expecting 392 features as input.')

In [ ]:
# Check submission file
import pandas as pd
df = pd.read_parquet('submission.parquet')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('First 10 rows:')
print(df.head(10))
print('Value counts:')
print(df['gesture'].value_counts())


Shape: (102, 2)
Columns: ['sequence_id', 'gesture']
First 10 rows:
  sequence_id                    gesture
0  SEQ_060707         Cheek - pinch skin
1  SEQ_037923              Text on phone
2  SEQ_028006   Forehead - pull hairline
3  SEQ_020034      Above ear - pull hair
4  SEQ_016979        Eyebrow - pull hair
5  SEQ_037140      Above ear - pull hair
6  SEQ_062937             Neck - scratch
7  SEQ_025391          Neck - pinch skin
8  SEQ_064204  Pull air toward your face
9  SEQ_063950         Cheek - pinch skin
Value counts:
gesture
Neck - scratch                                20
Cheek - pinch skin                            13
Neck - pinch skin                             13
Text on phone                                 11
Wave hello                                     7
Eyebrow - pull hair                            6
Eyelash - pull hair                            6
Above ear - pull hair                          5
Forehead - scratch                             5
Forehead - pull hai

In [ ]:
import pandas as pd
from src.utils.cmi_evaluation import calculate_cmi_score

# 例: 推論結果
df_pred = pd.read_parquet('submission.parquet')  # またはcsv等
# 例: 正解ラベル
df_true = pd.read_csv('../data/split/test_labels.csv')

# ラベル名が一致している前提
y_pred = df_pred['gesture'].values
y_true = df_true['gesture'].values

# 必要ならLabelEncoderでエンコード
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(list(y_true) + list(y_pred))
y_true_enc = le.transform(y_true)
y_pred_enc = le.transform(y_pred)

# CMIスコア計算
cmi_score, binary_f1, macro_f1, test_acc = calculate_cmi_score(y_pred_enc, y_true_enc, label_encoder=le, verbose=True)
print(f'CMI Score: {cmi_score:.4f}, Binary F1: {binary_f1:.4f}, Macro F1: {macro_f1:.4f}, Accuracy: {test_acc:.4f}')

CMI評価指標計算開始...
y_true shape: (102,), y_pred shape: (102,)
ラベル変換完了: 102 samples
データ中のジェスチャー: 18種類
Binary分類 - Target: 64, Non-Target: 38
Binary F1: 0.6963
Macro F1: 0.1124
CMI Score: 0.4044
Test Accuracy: 0.0588
CMI Score: 0.4044, Binary F1: 0.6963, Macro F1: 0.1124, Accuracy: 0.0588


Target / Non‑target 比率 (%)
sequence_type  Non-Target  Target
fold                             
0                    39.7    60.3
1                    40.4    59.6
2                    40.3    59.7
3                    40.4    59.6
4                    40.0    60.0
